# Kaggle environment and model cache only

Target notebook: https://www.kaggle.com/code/aohkvd/nl2sql-b1-b2-2026-09-22-r01
Use a private CPU setup copy, Internet enabled, Accelerator None. This notebook never invokes inference, encoding, training, BigQuery or a GPU probe. Retain its outputs privately. All mutating/acquisition flags default to False. Llama access must be approved on Hugging Face first; no token belongs in a cell or output.


In [ ]:
KIT_ROOT = "/kaggle/input/<private-setup-dataset>"
BUNDLE_ARCHIVE = "/kaggle/input/<private-input-dataset>/input-bundle.tar.gz"
BUNDLE_SHA256 = "2aa4dac0cb4110783279c3361b4c7618ce601402ed3a40cef168adad75caa6de"
WORKSPACE = "/kaggle/working/b12-input"
RUNTIME = "/kaggle/working/b12-runtime"
BOOTSTRAP = "/kaggle/working/b12-bootstrap"
HF_HOME = "/kaggle/working/hf-home"
ALLOW_INPUT_UNPACK = False
ALLOW_ENV_INSTALL = False
ALLOW_CACHE_DOWNLOADS = False
ALLOW_SECRET_ACCESS = False
CACHE_COMPONENT = "encoder"  # change to "all" only after Llama access approval


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
import tarfile

EXPECTED_SETUP_FILES = {'scripts/26_kaggle_environment.py': '91be7869ae7223b93e7ff6c3c7a58589b241161ef7a782351f02e105e86d9218', 'configs/kaggle-b12-model-pins.json': '6f72c03fe03c379b1ed0844192ef17c94cb60e908c5269f0e49750a32f04998a', 'configs/kaggle-b12-requirements.lock.txt': 'da3c09c04886fca688aa6a2e54aa1d7c46726be5a4c67b19bc2ba8b9ab36a5cd', 'configs/kaggle-b12-bootstrap.lock.txt': '021d7fd19573db49d3acd74056047912457790f20ceffb4d1196ebffa067a9c8'}
kit = Path(KIT_ROOT)
workspace = Path(WORKSPACE)
runtime_python = str(Path(RUNTIME) / "bin/python")
permitted = ("PATH", "LD_LIBRARY_PATH", "LANG", "LC_ALL")
setup_env = {name: os.environ[name] for name in permitted if name in os.environ}
setup_env.update(UV_CACHE_DIR="/kaggle/working/b12-uv-cache", UV_PYTHON_INSTALL_DIR="/kaggle/working/b12-python", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")

def check_setup_files():
    for name, expected in EXPECTED_SETUP_FILES.items():
        target = kit / name
        if target.is_symlink() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("Setup source fingerprint mismatch: " + name)

if not ALLOW_INPUT_UNPACK:
    print("BLOCKED: input unpack disabled")
else:
    check_setup_files()
    if not Path("/kaggle/working").is_dir():
        raise ValueError("Run this setup in Kaggle")
    archive = Path(BUNDLE_ARCHIVE)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != BUNDLE_SHA256:
        raise ValueError("Bilingual input archive SHA mismatch")
    if workspace.exists():
        raise ValueError("Use a fresh workspace; inputs are immutable")
    with tarfile.open(archive) as bundle:
        entries = bundle.getmembers()
        if len({m.name for m in entries}) != len(entries) or sum(m.size for m in entries) > 128*1024*1024:
            raise ValueError("Archive duplicates/size invalid")
        for item in entries:
            path = PurePosixPath(item.name)
            if not item.isfile() or path.is_absolute() or any(p in {".", ".."} or p.startswith(".") for p in item.name.split("/")):
                raise ValueError("Unsafe archive entry")
        bundle.extractall(workspace, filter="data")
    print("Unpacked accepted input bundle; free disk bytes:", shutil.disk_usage("/kaggle/working").free)


## Isolated environment

Enable installation only after verifying the setup files and fresh input bundle. Python 3.11.15 is separate from the notebook kernel; uv 0.11.14 and every runtime distribution are pinned with hashes. Do not change uv.lock or replace the research model to work around installation failures. Installation on this Kaggle session is not yet verified by the local handoff.


In [ ]:
if not ALLOW_ENV_INSTALL:
    print("BLOCKED: environment installation disabled")
else:
    check_setup_files()
    if not workspace.is_dir():
        raise ValueError("Unpack the input bundle first")
    subprocess.run([sys.executable, "-m", "pip", "--isolated", "install", "--no-deps", "--require-hashes", "--target", BOOTSTRAP, "-r", str(kit / "configs/kaggle-b12-bootstrap.lock.txt")], env=setup_env, check=True)
    uv_binary = str(Path(BOOTSTRAP) / "bin/uv")
    if not Path(RUNTIME).exists():
        subprocess.run([uv_binary, "--no-config", "venv", "--python", "3.11.15", "--managed-python", RUNTIME], env=setup_env, check=True)
    subprocess.run([uv_binary, "--no-config", "--no-cache", "pip", "sync", "--python", runtime_python, "--require-hashes", "--index-url", "https://pypi.org/simple", str(kit / "configs/kaggle-b12-requirements.lock.txt")], env=setup_env, check=True)
    subprocess.run([runtime_python, "--version"], env=setup_env, check=True)
    print("Installation command completed; still requires cache/runtime preflight")


## Exact pinned cache

Encoder is public. Llama is gated and this user's access is not yet requested. After approval only, add a read token in Kaggle Add-ons → Secrets with label HF_TOKEN and grant this private setup notebook access. Do not paste it into source. Enable secret access explicitly only for `CACHE_COMPONENT="all"` or `"model"`. Download runs in the isolated interpreter and passes the token through stdin, never argv or environment. The helper never logs in or saves a token. About 16.07 GB of model files plus 91.58 MB of encoder files are required, in addition to the runtime. No duplicate original/PyTorch/TF/ONNX weights are fetched. Public hashes check downloaded bytes; they do not prove load compatibility.


In [ ]:
if not ALLOW_CACHE_DOWNLOADS:
    print("BLOCKED: cache acquisition disabled")
else:
    check_setup_files()
    if CACHE_COMPONENT not in ("all", "encoder", "model"):
        raise ValueError("Invalid cache component")
    pins = json.loads((kit / "configs/kaggle-b12-model-pins.json").read_bytes())
    kinds = ("model", "encoder") if CACHE_COMPONENT == "all" else (CACHE_COMPONENT,)
    expected_bytes = sum(pins["models"][kind]["total_bytes"] for kind in kinds)
    if shutil.disk_usage("/kaggle/working").free < expected_bytes + 2*1024**3:
        raise ValueError("Insufficient free disk for selected cache plus download margin")
    argv = [runtime_python, str(kit / "scripts/26_kaggle_environment.py"), "fetch-cache", "--pins", str(kit / "configs/kaggle-b12-model-pins.json"), "--hf-home", HF_HOME, "--component", CACHE_COMPONENT, "--allow-downloads"]
    cache_env = dict(setup_env, HF_HOME=HF_HOME, HF_HUB_OFFLINE="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")
    if "model" in kinds:
        if not ALLOW_SECRET_ACCESS:
            raise ValueError("Llama requires explicit secret-access opt-in after approval")
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        try:
            process = subprocess.run([*argv, "--token-stdin"], input=token, text=True, env=cache_env, capture_output=True)
        finally:
            del token
    else:
        process = subprocess.run(argv, text=True, env=cache_env, capture_output=True)
    if process.returncode:
        raise RuntimeError("Cache acquisition blocked/failed; no raw authenticated stderr is displayed")
    cache_evidence = json.loads(process.stdout)
    evidence_path = Path("/kaggle/working/b12-cache-byte-verification.json")
    if evidence_path.exists() and json.loads(evidence_path.read_bytes()) != cache_evidence:
        raise ValueError("Preserve previous cache evidence and choose a fresh evidence path")
    evidence_path.write_text(json.dumps(cache_evidence, sort_keys=True, indent=2) + "\n")
    print(cache_evidence)


## Runtime preflight and handoff

Use the Python 3.11 runtime subprocess for every CLI17/CLI25 call, even if the Kaggle notebook kernel has another version. This cell generates EN/VI configs outside immutable inputs with genuine metadata revisions. Campaign and timeout stay null until the separately authorized smoke/run preparation; runtime remains blocked as intended. Capture notebook URL/version, CPU environment/install logs and cache byte evidence. Stop here: GPU smoke and genuine runs are separate stages. Do not execute the inference notebook unchanged under a different Python kernel.


In [ ]:
if not Path(runtime_python).is_file() or not workspace.is_dir():
    print("BLOCKED: isolated runtime and immutable inputs are required")
else:
    check_setup_files()
    pins = json.loads((kit / "configs/kaggle-b12-model-pins.json").read_bytes())
    for language in ("en", "vi"):
        config = json.loads((workspace / "configs/kaggle-b12-run.template.json").read_bytes())
        config.update(language=language, model_revision=pins["models"]["model"]["revision"], encoder_revision=pins["models"]["encoder"]["revision"], hf_home=HF_HOME)
        path = Path("/kaggle/working") / f"b12-{language}-environment-config.json"
        if path.exists() and json.loads(path.read_bytes()) != config:
            raise ValueError("Preserve previous config; choose a fresh output")
        path.write_text(json.dumps(config, sort_keys=True, indent=2) + "\n")
    print("Pinned config files generated; campaign/time cap remain unresolved, inference disabled")
